# Prepare Public Monthly Summary - No Publication
Optional separate notebook. Import 12 beside this and keep existing 09 in
the same folder. Run only after ETL completes with no concurrent writer.
Reads operational/check metadata, not raw trip tables. Outputs approved
aggregate JSON only; no upload, credentials, or business-table writes.
Latest incomplete/failed attempts block the export instead of being hidden.

In [0]:
%run ./09_runtime_sources

In [0]:
%run ./12_public_summary_sources

In [0]:
from datetime import datetime, timezone
from zoneinfo import ZoneInfo
from pyspark.sql import functions as F
from pyspark.sql.window import Window

dbutils.widgets.text('catalog_name','workspace','Catalog')
dbutils.widgets.text('schema_name','nyc_taxi_lakehouse','Schema')
dbutils.widgets.text('start_period','2024-01','First source period')
dbutils.widgets.text('end_period','','Last period; blank = latest due')
dbutils.widgets.text('boundary_grace_days','1','Boundary grace days')
catalog, schema = dbutils.widgets.get('catalog_name'), dbutils.widgets.get('schema_name')
if not all(re.fullmatch(r'[A-Za-z_][A-Za-z0-9_]*',name) for name in (catalog,schema)):
    raise ValueError('Catalog and schema must be simple identifiers')
start = dbutils.widgets.get('start_period')
end = dbutils.widgets.get('end_period') or due_period(datetime.now(ZoneInfo('Asia/Kolkata')).date(),2)
_months(start,end)
policy = policy_identity(COMPLETION_POLICY_DIGEST,int(dbutils.widgets.get('boundary_grace_days')))
event_table = f'{catalog}.{schema}.pipeline_month_events'
check_table = f'{catalog}.{schema}.pipeline_reconciliation_checks'
events_df = spark.table(event_table).where(F.col('source_period').between(start,end))
event_window = Window.partitionBy('source_period').orderBy(F.col('event_time_utc').desc(),F.col('event_id').desc())
latest_events = events_df.withColumn('_rn',F.row_number().over(event_window)).where('_rn = 1').drop('_rn')
checks_df = spark.table(check_table).withColumn('source_period',F.concat_ws('-',F.col('process_year'),F.lpad('process_month',2,'0')))
checks_df = checks_df.where(F.col('source_period').between(start,end))
check_window = Window.partitionBy('source_period').orderBy(F.col('reconciled_at_utc').desc())
latest_checks = checks_df.withColumn('_rank',F.dense_rank().over(check_window)).where('_rank = 1').drop('_rank')
# Bound driver collection. Changed schemas/oversized evidence must be reviewed.
limit_months = len(_months(start,end))
events = [row.asDict() for row in latest_events.limit(limit_months+1).collect()]
checks = [row.asDict() for row in latest_checks.limit(limit_months*49+1).collect()]
public_snapshot = build_snapshot(checks,events,policy_version=policy,start=start,end=end,
                                 generated_at_utc=datetime.now(timezone.utc).isoformat())
print(json.dumps(public_snapshot,indent=2))
print('PREPARED ONLY: no public upload or Job change was performed.')